In [55]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [56]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)


(array([0, 1]), array([900, 100]))

In [57]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


In [ ]:
EXPERIMENT 1 :- TRAIN LOGISTIC REGRESSION CLASSIFIER

In [58]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



In [ ]:
EXPERIMENT 2 :- TRAIN RANDOM FOREST CLASSIFIER

In [59]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))


              precision    recall  f1-score   support

           0       0.97      0.99      0.98       270
           1       0.91      0.70      0.79        30

    accuracy                           0.96       300
   macro avg       0.94      0.85      0.89       300
weighted avg       0.96      0.96      0.96       300



In [ ]:
EXPERIMENT 3 :- TRAIN XGBOOST

In [60]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



In [ ]:
Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [61]:
!pip install imbalanced-learn

In [62]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)


(array([0, 1]), array([619, 619]))

In [63]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [ ]:
TRACKING EXPERIMENTS USING MLFLOW

In [64]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [65]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)


In [66]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [47]:
mlflow.set_experiment("Anomaly Detection-v3")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):
        
        mlflow.log_params(params)
        
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })         
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/02/27 11:55:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/27 11:55:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/4/runs/e5e4e661ad4d4841b4b32d86bd191550
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/02/27 11:55:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/27 11:55:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://localhost:5000/#/experiments/4/runs/5ed8935a21c8442c8b439a66036a3f59
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/02/27 11:56:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/4/runs/caa2d8a1d7d94fdfac50da2195e675eb
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/02/27 11:56:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/4/runs/cd8bcde531f64b5a9436ab2683567065
🧪 View experiment at: http://localhost:5000/#/experiments/4


In [ ]:
REGISTER THE MODEL

In [49]:
model_name = 'XGB-Smote'
run_id = input('Please type RunID')

model_uri = f'runs:/{run_id}/model'

mlflow.register_model(model_uri=model_uri, name=model_name)

Please type RunID ee58faef03a848588c53b53843af766e


Successfully registered model 'XGB-Smote'.
2026/02/27 12:34:28 WARNING mlflow.tracking._model_registry.fluent: Run with id ee58faef03a848588c53b53843af766e has no artifacts at artifact path 'model', registering model based on models:/m-20db1d0f9acb400d82b0394c8b56b3c7 instead
2026/02/27 12:34:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


<ModelVersion: aliases=[], creation_timestamp=1772175868249, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1772175868249, metrics=None, model_id=None, name='XGB-Smote', params=None, run_id='ee58faef03a848588c53b53843af766e', run_link='', source='models:/m-20db1d0f9acb400d82b0394c8b56b3c7', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [ ]:
LOAD THE MODEL

In [50]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [ ]:
Transitioning the model to production

In [51]:
current_model_uri = f"models:/{model_name}@challenger"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1772177645852, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1772177645852, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='ee58faef03a848588c53b53843af766e', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [67]:
pip install dagshub

Note: you may need to restart the kernel to use updated packages.


In [68]:
import dagshub
dagshub.init(repo_owner='unnati.c', repo_name='mlflow_dags_hub', mlflow=True)


Initialized MLflow to track repo "unnati.c/mlflow_dags_hub"

Repository unnati.c/mlflow_dags_hub initialized!

In [69]:

# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
# mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  


2026/03/02 07:50:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 07:50:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/2/runs/211336e409004d5490bb3d5a6cb45e3d
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/03/02 07:51:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 07:51:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://localhost:5000/#/experiments/2/runs/ef05de8d844140408b1f8ed67f69aada
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/03/02 07:51:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/2/runs/ea423279351e4e1fb82583629e09dcca
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/03/02 07:51:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/2/runs/c14c7ea23c2d4a8e89b3c4cd2c4136e2
🧪 View experiment at: http://localhost:5000/#/experiments/2
